# 🚀 ARES: Colab T4 Interactive Audit & Benchmark Evaluation
**Adaptive Reliability with Expert Specialization (IEEE TNNLS)**

This notebook is optimized to run inside **VS Code connected to a Google Colab T4 GPU runtime** (or directly in Google Colab).

### 📋 Purpose & Verification Workflow:
1. **Environment & Hardware Audit**: Confirms NVIDIA T4 GPU (16 GB VRAM), CUDA acceleration, and required libraries (`transformers`, `peft`, `bitsandbytes`, `datasets`).
2. **Interactive 20-Sample GSM8K Spot Check (Qwen2.5-7B-Instruct 4-bit)**: Direct side-by-side comparison between **Base Model** and **Math Expert Adapter**:
   - Confirms that the LoRA adapter is genuinely active and diverges from the Base Model (no size mismatch, no silent fallback).
   - Verifies that the multi-tier regex answer extractor (`extract_math_answer`) accurately extracts the final CoT answer with 256 tokens.
3. **Empirical Benchmark Evaluation**: Re-runs the full 5-domain evaluation (GSM8K, MBPP, AI2-ARC, CommonsenseQA, WikiText-103) across all baseline strategies with `--max_new_tokens 256` and smart caching.
4. **Paper Table I & Statistical Output**: Formats the final empirical numbers directly into LaTeX code ready for the paper.

In [ ]:
# === [1/5] Hardware & Environment Verification ===
import os
import sys
import torch

print('=' * 65)
print('  ARES HARDWARE & ENVIRONMENT AUDIT')
print('=' * 65)

# 1. Verify CUDA & GPU
cuda_available = torch.cuda.is_available()
print(f'CUDA Available: {cuda_available}')
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device:     {gpu_name} ({vram_gb:.2f} GB VRAM)')
else:
    print('WARNING: CUDA is not active! Please select a GPU runtime (Runtime -> Change runtime type -> T4 GPU).')

# 2. Install / Verify Core Dependencies
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.41.0' 'peft>=0.12.0' 'accelerate>=0.30.0' 'bitsandbytes>=0.43.0' datasets scipy tabulate

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
print('\n✅ Environment packages verified and ready!')

In [ ]:
# === [2/5] Repository & Checkpoint Setup ===
import os
import sys
import shutil
import zipfile
from pathlib import Path

# 1. Ensure working directory is the repository root and up to date
if Path('src/ares').exists():
    repo_root = Path('.').resolve()
    print(f'Using current workspace: {repo_root}')
    !git pull origin main
elif Path('ARES-research/src/ares').exists():
    repo_root = Path('ARES-research').resolve()
    os.chdir(repo_root)
    print(f'Switched into cloned repository: {repo_root}')
    !git pull origin main
else:
    print('Cloning ARES research repository from GitHub...')
    !git clone https://github.com/sharksurfauto-byte/ARES-research.git
    os.chdir('ARES-research')
    repo_root = Path('.').resolve()
    print(f'Cloned and switched into: {repo_root}')

if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

ckpt_dir = repo_root / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

# 2. Check if a zip archive was uploaded to /content or workspace
zip_candidates = [
    Path('checkpoints.zip'),
    Path('/content/checkpoints.zip'),
    Path('../checkpoints.zip'),
    Path('outputs/checkpoints.zip'),
]
for z_path in zip_candidates:
    if z_path.exists():
        print(f'Found checkpoint archive at {z_path}! Extracting into {ckpt_dir}...')
        with zipfile.ZipFile(z_path, 'r') as zf:
            zf.extractall(ckpt_dir)
        print('✅ Extraction complete!')
        break

# 3. Check for existing mounted directories (Kaggle or Google Drive)
if not (ckpt_dir / 'reliability' / 'grm.pt').exists():
    candidate_dirs = [
        Path('/kaggle/input/datasets/aliasgharjjawadwala/ares-eval-input/checkpoints'),
        Path('/kaggle/input/ares-eval-input/checkpoints'),
        Path('/content/drive/MyDrive/checkpoints'),
        Path('/content/drive/MyDrive/ARES/checkpoints'),
    ]
    for cd in candidate_dirs:
        if (cd / 'reliability' / 'grm.pt').exists():
            print(f'Found checkpoints at {cd}! Copying into {ckpt_dir}...')
            shutil.copytree(cd, ckpt_dir, dirs_exist_ok=True)
            break

# 4. Optional: Kaggle API Credentials (if dataset is private)
KAGGLE_USERNAME = ''  # e.g., 'aliasgharjjawadwala'
KAGGLE_KEY = ''       # e.g., 'your_kaggle_api_key'

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY

if not (ckpt_dir / 'reliability' / 'grm.pt').exists():
    try:
        import kagglehub
        print("Attempting download via kagglehub ('aliasgharjjawadwala/ares-eval-input')...")
        path = kagglehub.dataset_download('aliasgharjjawadwala/ares-eval-input')
        src_p = Path(path) / 'checkpoints' if (Path(path) / 'checkpoints').exists() else Path(path)
        shutil.copytree(src_p, ckpt_dir, dirs_exist_ok=True)
        print(f'✅ Downloaded checkpoints to {ckpt_dir}')
    except Exception as e:
        print(f'\n[Kaggle Notice] Automated download requires either public dataset access or API credentials: {e}')

# 5. Interactive Upload Fallback if checkpoints are still missing
if not (ckpt_dir / 'reliability' / 'grm.pt').exists():
    print('\n' + '=' * 65)
    print('  CHECKPOINTS SETUP (Choose the quickest option):')
    print('=' * 65)
    print('  • Option A (Fastest): Locate "outputs/checkpoints.zip" on your local PC (79 MB).')
    print('    Drag and drop it into the Colab file panel on the left, or use the file dialog below.')
    print('  • Option B: In Kaggle (kaggle.com/datasets/aliasgharjjawadwala/ares-eval-input),')
    print('    go to Settings -> Visibility -> set to "Public". Then re-run this cell!')
    print('  • Option C: Set KAGGLE_USERNAME and KAGGLE_KEY above with your API token.')
    print('=' * 65)
    try:
        from google.colab import files
        print('\nOpening file uploader dialog... Select "outputs/checkpoints.zip" from your PC:')
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.zip'):
                print(f'Extracting uploaded {fn} into {ckpt_dir}...')
                with zipfile.ZipFile(fn, 'r') as zf:
                    zf.extractall(ckpt_dir)
                print('✅ Extraction complete!')
    except Exception as err:
        print(f'File upload prompt notice: {err}')

# 6. Verify Critical Checkpoints
assert (ckpt_dir / 'reliability' / 'grm.pt').exists(), (
    'ERROR: grm.pt missing! Upload outputs/checkpoints.zip to Colab, or make your Kaggle dataset Public.'
)
assert (ckpt_dir / 'reliability' / 'lrm.pt').exists(), 'ERROR: lrm.pt missing!'
router_ok = (ckpt_dir / 'router' / 'router_best.pt').exists() or (ckpt_dir / 'router' / 'router.pt').exists()
assert router_ok, 'ERROR: router checkpoint missing in checkpoints/router/!'

print('\n✅ ALL CRITICAL CHECKPOINTS VERIFIED:')
for p in sorted(ckpt_dir.rglob('*.pt')):
    print(f'   • {p.relative_to(repo_root)} ({p.stat().st_size / (1024*1024):.2f} MB)')

In [ ]:
# === [3/5] Interactive 20-Sample GSM8K Diagnostic Spot Check ===
# Evaluates Qwen2.5-7B-Instruct (4-bit NF4) with hidden dimension 3584 matching checkpoints!

import torch
import pandas as pd
from tabulate import tabulate
from ares.pipeline.ares_pipeline import ARESPipeline, PipelineConfig
from ares.data.benchmark_loader import load_gsm8k_samples, extract_math_answer, evaluate_prediction

# 1. Initialize Pipeline with Qwen2.5-7B-Instruct (dim 3584 matching checkpoints)
# On Colab T4 GPU, 7B runs in 4-bit NF4 quantization using ~4.8 GB VRAM.
config = PipelineConfig(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    checkpoints_dir='checkpoints',
    max_new_tokens=256,   # Generous token budget for full step-by-step reasoning
    do_sample=False,      # Deterministic greedy decoding
)

print('[ARES] Loading Qwen2.5-7B-Instruct (4-bit NF4, dim 3584) & Checkpoints...')
pipeline = ARESPipeline(config=config)
print(f'[ARES] Pipeline successfully loaded on device: {pipeline.device} (Hidden Dim: {pipeline.config.hidden_dim})')

# 2. Load 20 GSM8K test samples
print('\n[ARES] Loading 20 GSM8K test queries...')
spot_samples = load_gsm8k_samples(n_samples=20, split='test')

records = []
base_correct_count = 0
expert_correct_count = 0
diverged_count = 0

print('Evaluating samples (Base vs Math Expert with max_new_tokens=256)...\n')
for i, sample in enumerate(spot_samples):
    prompt = sample.prompt
    target_clean = sample.target_answer
    target_num = extract_math_answer(target_clean) or target_clean

    # A. Generate with Base Model
    res_base = pipeline.generate(prompt=prompt, strategy='base', max_new_tokens=256)
    base_text = res_base.generated_text
    base_ext = extract_math_answer(base_text)
    base_corr = evaluate_prediction(base_text, target_clean, eval_type='math_numeric')
    if base_corr:
        base_correct_count += 1

    # B. Generate with Math Expert Adapter
    res_expert = pipeline.generate(prompt=prompt, strategy='fixed_math', max_new_tokens=256)
    expert_text = res_expert.generated_text
    expert_ext = extract_math_answer(expert_text)
    expert_corr = evaluate_prediction(expert_text, target_clean, eval_type='math_numeric')
    if expert_corr:
        expert_correct_count += 1

    # C. Divergence Check
    diverged = (base_text != expert_text)
    if diverged:
        diverged_count += 1

    records.append({
        '#': i + 1,
        'Question': (prompt[:50] + '...') if len(prompt) > 50 else prompt,
        'Target': target_num,
        'Base Extracted': base_ext if base_ext is not None else 'None',
        'Base Ok': '✅' if base_corr else '❌',
        'Expert Extracted': expert_ext if expert_ext is not None else 'None',
        'Expert Ok': '✅' if expert_corr else '❌',
        'Diverged?': '✅ YES' if diverged else '❌ IDENTICAL',
    })

# 3. Display Results Table
df_spot = pd.DataFrame(records)
print(tabulate(df_spot, headers='keys', tablefmt='github', showindex=False))

base_acc = (base_correct_count / len(spot_samples)) * 100.0
expert_acc = (expert_correct_count / len(spot_samples)) * 100.0
div_rate = (diverged_count / len(spot_samples)) * 100.0

print('\n' + '=' * 65)
print('  GSM8K 20-SAMPLE DIAGNOSTIC AUDIT SUMMARY')
print('=' * 65)
print(f'  • Base Model Accuracy:     {base_acc:.1f}% ({base_correct_count}/{len(spot_samples)})')
print(f'  • Math Expert Accuracy:    {expert_acc:.1f}% ({expert_correct_count}/{len(spot_samples)})')
print(f'  • Generation Divergence:   {div_rate:.1f}% ({diverged_count}/{len(spot_samples)})')
print(f'  • Base Numbers Extracted:  {sum(1 for r in records if r["Base Extracted"] != "None")}/{len(spot_samples)}')
print(f'  • Expert Numbers Extracted:{sum(1 for r in records if r["Expert Extracted"] != "None")}/{len(spot_samples)}')
print('=' * 65)

assert div_rate >= 80.0, (
    f'SANITY WARNING: Divergence rate is only {div_rate:.1f}%. '
    'Expected Base and Adapter completions to genuinely diverge!'
)
print('\n✅ SPOT CHECK PASSED: LoRA Expert is actively modifying generations and extractor is parsing answers!')

In [ ]:
# === [4/5] Full Multi-Domain Benchmark Evaluation (7B 4-bit Backbone) ===
# Evaluates all 5 domains across baseline strategies with Smart Caching.

from ares.data.benchmark_loader import load_all_benchmark_samples
from ares.pipeline.baselines import BaselineComparator
from ares.pipeline.metrics import MetricsCalculator

# Configure evaluation scale:
# 50 per domain = 250 queries total (~15-20 minutes on T4 GPU)
# 100 per domain = 500 queries total (~35 minutes on T4 GPU)
SAMPLES_PER_DOMAIN = 50
SPLIT = 'test'
MAX_NEW_TOKENS = 256

print(f'[ARES Data] Loading {SAMPLES_PER_DOMAIN} benchmark samples per domain (split={SPLIT})...')
samples_dict = load_all_benchmark_samples(n_samples_per_domain=SAMPLES_PER_DOMAIN, split=SPLIT)
all_eval_samples = []
for domain_name, s_list in samples_dict.items():
    print(f'  • {domain_name:<10}: {len(s_list)} samples')
    all_eval_samples.extend(s_list)

print(f'\nTotal test queries to evaluate: {len(all_eval_samples)}')

comparator = BaselineComparator(
    pipeline=pipeline,
    strategies=['BASE', 'FIXED_EXPERT', 'DYNAMIC_ARES', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER'],
    fixed_expert='math',
    threshold=0.5,
)

def on_progress(batch_results, current, total):
    if current % 10 == 0 or current == total:
        pct = (current / total) * 100.0
        print(f'  -> Evaluated {current}/{total} samples ({pct:.1f}%)...', flush=True)

print('\nStarting baseline evaluation across all 5 strategies...')
eval_results = comparator.evaluate_batch(
    all_eval_samples,
    max_new_tokens=MAX_NEW_TOKENS,
    checkpoint_callback=on_progress,
)

metadata = {
    'model_name': config.model_name,
    'samples_per_domain': SAMPLES_PER_DOMAIN,
    'split': SPLIT,
    'max_new_tokens': MAX_NEW_TOKENS,
}

report = MetricsCalculator.calculate_metrics(eval_results, metadata=metadata)
report.print_summary()

os.makedirs('outputs', exist_ok=True)
report.save_json('outputs/benchmark_results_7b.json')
print('\n✅ Saved evaluation results to outputs/benchmark_results_7b.json')

[ARES Data] Loading 50 benchmark samples per domain (split=test)...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

[WikiText] HF loading fallback: Invalid HF URI 'hf://datasets/wikitext@b08601e04326c79dfdd32d625aee71d232d685c3/.huggingface.yaml'. Repository id must be 'namespace/name', got 'wikitext'.
[GSM8K] HF loading fallback: Invalid HF URI 'hf://datasets/gsm8k@740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml'. Repository id must be 'namespace/name', got 'gsm8k'.


README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

[MBPP] HF loading fallback: Invalid HF URI 'hf://datasets/mbpp@4bb6404fdc6cacfda99d4ac4205087b89d32030c/.huggingface.yaml'. Repository id must be 'namespace/name', got 'mbpp'.


README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

[AI2-ARC] HF loading fallback: Invalid HF URI 'hf://datasets/ai2_arc@210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml'. Repository id must be 'namespace/name', got 'ai2_arc'.


README.md:   0%|          | 0.00/7.39k [00:00<?, ?B/s]

[Reasoning] HF loading fallback: Invalid HF URI 'hf://datasets/commonsense_qa@94630fe30dad47192a8546eb75f094926d47e155/.huggingface.yaml'. Repository id must be 'namespace/name', got 'commonsense_qa'.
  • general   : 50 samples
  • math      : 50 samples
  • code      : 50 samples
  • science   : 50 samples
  • reasoning : 50 samples

Total test queries to evaluate: 250

Starting baseline evaluation across all 5 strategies...
[ARES Baselines] Processing sample 1/250 (domain: general)...


In [ ]:
# === [5/5] Table I Formatter & Statistical Significance Tests ===
# Formats the empirical run into LaTeX rows for Table I and computes statistical tests.

import numpy as np
from scipy import stats

print('=' * 75)
print('  EMPIRICAL TABLE I (LATEX ROWS)')
print('=' * 75)

domains = ['math', 'code', 'science', 'reasoning', 'general']
strategies = ['BASE', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER', 'FIXED_EXPERT', 'DYNAMIC_ARES']

display_names = {
    'BASE': 'Base Qwen2.5-7B (Zero-Shot)',
    'THRESHOLD_ROUTER': 'Entropy / Threshold (7B)',
    'RANDOM_ROUTER': 'Random Router (7B)',
    'FIXED_EXPERT': 'Fixed Expert (Math 7B)',
    'DYNAMIC_ARES': '\\textbf{ARES 7B (Learned Router)}',
}

latex_rows = []
for strat in strategies:
    strat_results = [r for r in eval_results if strat in r.results]
    n_total = len(strat_results)

    dom_accs = {}
    for d in domains:
        d_samples = [r for r in strat_results if r.domain == d]
        if d_samples:
            acc = sum(1 for r in d_samples if r.correctness.get(strat, False)) / len(d_samples) * 100.0
            dom_accs[d] = acc
        else:
            dom_accs[d] = 0.0

    overall_acc = sum(1 for r in strat_results if r.correctness.get(strat, False)) / n_total * 100.0 if n_total > 0 else 0.0
    inv_rate = sum(1 for r in strat_results if r.expert_invocations.get(strat, False)) / n_total * 100.0 if n_total > 0 else 0.0
    savings = 100.0 - inv_rate
    mean_lat = np.mean([r.latencies_ms.get(strat, 0.0) for r in strat_results])

    row_str = (
        f"{display_names.get(strat, strat):<35} & "
        f"{dom_accs['math']:>5.1f}\\% & "
        f"{dom_accs['code']:>5.1f}\\% & "
        f"{dom_accs['science']:>5.1f}\\% & "
        f"{dom_accs['reasoning']:>5.1f}\\% & "
        f"{dom_accs['general']:>5.1f}\\% & "
        f"{overall_acc:>6.2f}\\% & "
        f"{inv_rate:>5.1f}\\% & "
        f"{savings:>5.1f}\\% & "
        f"{mean_lat:>7.1f} ms \\\\"
    )
    latex_rows.append(row_str)

print('\n'.join(latex_rows))
print('=' * 75)

# Statistical Significance Testing: Base vs Dynamic ARES
base_correct = [int(r.correctness.get('BASE', False)) for r in eval_results]
ares_correct = [int(r.correctness.get('DYNAMIC_ARES', False)) for r in eval_results]

t_stat, p_val = stats.ttest_rel(ares_correct, base_correct)
print(f'\nStatistical Significance (ARES vs Base):')
print(f'  • Paired Student t-statistic: t = {t_stat:.2f}, p-value = {p_val:.4e}')

# McNemar test for paired binary classification
contingency = np.zeros((2, 2))
for b, a in zip(base_correct, ares_correct):
    contingency[b, a] += 1
b_wrong_a_right = contingency[0, 1]
b_right_a_wrong = contingency[1, 0]
chi2 = ((abs(b_wrong_a_right - b_right_a_wrong) - 1)**2) / (b_wrong_a_right + b_right_a_wrong + 1e-8)
print(f'  • McNemar chi-square:        chi2 = {chi2:.2f} (discordants: {int(b_wrong_a_right)} ARES-only vs {int(b_right_a_wrong)} Base-only)')